# Parameter Profiles: Depth-Dependent Parameters

This notebook demonstrates the **parameter profile** feature, which lets a spectral
parameter vary continuously along an auxiliary physical axis (here: sample depth)
and averages the resulting spectra over that axis.

**Example System:** Semiconductor core level (e.g. Si 2p) measured with
time-resolved XPS after laser excitation.

**Two depth profiles are combined:**
1. **IMFP amplitude profile** (`GLP_01_A` via `pExpDecay`):  
   Signal from depth *z* decays as `A * exp(-z / lambda)`.
2. **Band bending position profile** (`GLP_01_x0` via `pLinear`):  
   Core level binding energy shifts linearly with depth: `x0(z) = x0_surface + m * z`.

**Time dependence (spectral diffusion):**  
After laser excitation the band bending gradient `m` recovers exponentially.
This is modelled by making the profile parameter `pLinear_01_m` time-dependent.

**Workflow:**
1. Load and inspect data
2. Fit baseline: standard (no profiles) vs. profile-aware
3. Slice-by-Slice fitting with profiles
4. Global 2D fitting with profiles + time-dependent band bending

Data generated by `generate_data.ipynb` (see that notebook for physics details).

In [ ]:
import os

import numpy as np

import trspecfit
from trspecfit.utils.lmfit import MC

## 1. Load Data

Load synthetic time- and energy-resolved XPS data from CSV files.
The auxiliary axis (`aux_axis`) represents probing depth in nm.

In [ ]:
# Create project
project = trspecfit.Project(path=os.getcwd())

In [ ]:
# Load data from CSV files
data_folder = "data"

file = trspecfit.File(
    parent_project=project,
    path=data_folder,
    data=np.loadtxt(
        project.path / data_folder / "data.csv", delimiter=","
    ),
    energy=np.loadtxt(project.path / data_folder / "energy.csv"),
    time=np.loadtxt(project.path / data_folder / "time.csv"),
    aux_axis=np.loadtxt(project.path / data_folder / "aux_axis.csv"),
)

print(f"Data shape: {file.data.shape}")
print(f"Energy range: {file.energy.min():.1f} - {file.energy.max():.1f} eV")
print(f"Time range: {file.time.min():.0f} - {file.time.max():.0f} ps")
print(f"Aux axis (depth): {file.aux_axis.min():.0f} - {file.aux_axis.max():.0f} nm  ({len(file.aux_axis)} points)")

## 2. Inspect Data

Visualize the raw data to identify features and appropriate fitting regions.

In [ ]:
file.describe()

## 3. Define Fitting Region

Set energy and time limits, and extract the baseline (pre-pump) spectrum.

In [ ]:
file.set_fit_limits(
    energy_limits=[95.5, 101.5],
    time_limits=[-50, 300],
)

# Average pre-pump spectra as the baseline
file.define_baseline(
    time_start=-50,
    time_stop=-10,
    time_type="abs",
)

## 4. Fit Baseline Spectrum

### 4a. Standard baseline (no profiles)

First we try a conventional GLP fit without depth profiles.
This gives **effective** (depth-averaged) parameters and serves as a comparison.

In [ ]:
file.load_model(
    model_yaml="models_energy.yaml",
    model_info="base",
)

file.describe_model(model_info="base", detail=0)

In [ ]:
file.fit_baseline(model_name="base", stages=2)

# The fitted A (~30) and x0 (~99.4 eV) are depth-averaged effective values,
# NOT the true surface amplitude (100) or surface binding energy (99.5 eV).
# Look at the residual: the asymmetric shape hints at depth structure.

### 4b. Profile-aware baseline

The residual above shows structure that a single depth-averaged peak cannot capture.
Now we add depth profiles so the fit can resolve the underlying physics:

| Profile | Parameter | Function | Physical meaning |
|---|---|---|---|
| IMFP | `GLP_01_A` | `pExpDecay(z, A, tau)` | Surface sensitivity |
| Band bending | `GLP_01_x0` | `pLinear(z, m, b)` | Built-in electric field |

The energy model parameter `A` is fixed to 0; the `pExpDecay` profile provides
the full depth-integrated amplitude.  
`x0` is the **surface** binding energy; the `pLinear` profile adds the depth shift.

See `models_profile.yaml` for the YAML definitions.

In [ ]:
# Load the profile-aware baseline model (A=0 fixed)
file.load_model(
    model_yaml="models_energy.yaml",
    model_info="profile_base",
)

# Attach IMFP amplitude profile
file.add_par_profile(
    target_model="profile_base",
    target_parameter="GLP_01_A",
    profile_yaml="models_profile.yaml",
    profile_model=["GLP_01_A"],
)

# Attach band bending position profile
file.add_par_profile(
    target_model="profile_base",
    target_parameter="GLP_01_x0",
    profile_yaml="models_profile.yaml",
    profile_model=["GLP_01_x0"],
)

file.describe_model(model_info="profile_base", detail=0)

In [ ]:
# Sanity check: inspect the GLP component and its depth profiles
comp = file.model_active.components[1]  # GLP component
comp.describe(detail=1)
comp.plot(plot_every=3)  # plot every 3rd depth trace
comp.plot(plot_traces=False)  # depth-averaged spectrum

In [ ]:
file.fit_baseline(model_name="profile_base", stages=2)

# Now we recover physically meaningful parameters:
#   pExpDecay_01_tau  ~ 2.0 nm    (IMFP)
#   pLinear_01_m       ~ -0.5 eV/nm (equilibrium band bending)
#   GLP_01_x0         ~ 99.5 eV   (surface binding energy)

## 5. Slice-by-Slice Fitting

Fit each time slice independently with profiles attached.
This reveals how the surface binding energy `x0` evolves over time
and helps identify which parameters need explicit time-dependence.

In [ ]:
# Load SbS model (same structure as 2D, but no time dynamics)
file.load_model(
    model_yaml="models_energy.yaml",
    model_info="SbS",
)

# Attach the same profiles
file.add_par_profile(
    target_model="SbS",
    target_parameter="GLP_01_A",
    profile_yaml="models_profile.yaml",
    profile_model=["GLP_01_A"],
)
file.add_par_profile(
    target_model="SbS",
    target_parameter="GLP_01_x0",
    profile_yaml="models_profile.yaml",
    profile_model=["GLP_01_x0"],
)

file.describe_model("SbS", detail=0)

In [ ]:
file.fit_slice_by_slice(
    model_name="SbS",
    stages=1,
    try_ci=0,
)

## 6. Global 2D Fitting

The band bending gradient `pLinear_01_m` is itself time-dependent:
after laser excitation it collapses and recovers exponentially.

We attach a `Dynamics` model to the **profile model's parameter** `pLinear_01_m`
(not to a parameter of the energy model). This enables *spectral diffusion*:
the depth profile shape changes at every time step.

```
energy model         profile model         dynamics model
GLP_01_x0  ---p_vary--> pLinear_01_m  ---t_vary--> BandBendingRecovery
                        (eV/nm)                   (gaussCONV + expFun)
```

We use the high-level API:
1. `add_par_profile` attaches the profile (its parameters become searchable).
2. `add_time_dependence` finds `GLP_01_x0_pLinear_01_m` inside the attached
   profile and adds the dynamics; `model.dim` is promoted to 2 automatically.

See `models_time.yaml` for the dynamics YAML definition.

In [ ]:
# Load 2D energy model
file.load_model(
    model_yaml="models_energy.yaml",
    model_info="2D",
)

# Attach IMFP profile (static)
file.add_par_profile(
    target_model="2D",
    target_parameter="GLP_01_A",
    profile_yaml="models_profile.yaml",
    profile_model=["GLP_01_A"],
)

# Attach band bending profile (its gradient will be time-dependent)
file.add_par_profile(
    target_model="2D",
    target_parameter="GLP_01_x0",
    profile_yaml="models_profile.yaml",
    profile_model=["GLP_01_x0"],
)

# Add time dependence to the profile's gradient parameter
file.add_time_dependence(
    target_model="2D",
    target_parameter="GLP_01_x0_pLinear_01_m",
    dynamics_yaml="models_time.yaml",
    dynamics_model=["BandBendingRecovery"],
)

file.describe_model(model_info="2D", detail=0)

In [ ]:
mc_settings = MC(
    use_mc=0,  # Set to 1 for Monte Carlo uncertainty estimation
    steps=5000,
    nwalkers=20,
    thin=1,
)

file.fit_2d(
    model_name="2D",
    stages=2,
    try_ci=0,
    mc_settings=mc_settings,
)

## Tips for Profile Fitting

**Profile basics:**
- Set `aux_axis` once in `File(...)`; it propagates to all loaded models.
- Profile model name in YAML must match the target energy model parameter
  exactly (e.g. `GLP_01_A`).
- Use `file.add_par_profile(target_model, target_parameter, profile_yaml, profile_model)` for profiles
  (analogous to `file.add_time_dependence`).

**Parameter naming:**
- Profile lmfit parameters are prefixed with the profile model name:
  `<profile_model_name>_<component>_<par>`, e.g. `GLP_01_A_pExpDecay_01_A`.

**Base parameter and profile:**
- Total value at aux point *i*: `base_value + p_model.value_1d[i]`.
- For IMFP amplitude: set `A = 0` (fixed); `pExpDecay` provides the full amplitude.
- For band bending: set `x0` to the surface value; `pLinear` adds the depth shift.

**Time-dependent profiles:**
1. Call `add_par_profile(...)` first to make profile parameters searchable.
2. Call `add_time_dependence(..., target_parameter=<profile_par>)` — it searches
   the energy model first, then attached profiles. `model.dim` is auto-promoted.

**Multiple profiles on one component:**
- Multiple parameters can each have their own profile sharing the same `aux_axis`.
- They are evaluated jointly at each aux point (correlated variation).

**Profile functions (`fcts_profile`):**
- `pExpDecay(x, A, tau)` — IMFP weighting, Beer-Lambert
- `pLinear(x, m, b)` — band bending, linear gradients
- `pGauss(x, A, x0, SD)` — Gaussian broadening, fluence averaging

**Next steps:**
- Combine with `02_dependent_parameters` (spin-orbit split peak with profiles).
- For fluence-averaging: use `pGauss` profile on A with `aux_axis = fluence_values`.